# Refund Policy Agent

This agent handles customer inquiries about:
- Refund policies
- Return procedures
- Exchange requests
- Warranty claims

## Setup

In [11]:
import sys
!{sys.executable} -m pip install -U langchain_openai langgraph openai

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("Chatbot")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence
import operator

model_name = "gpt-4o-mini"
llm = ChatOpenAI(model=model_name)

print("✅ Setup complete")

✅ Setup complete


## Define Agent State

In [12]:
class AgentState(TypedDict):
    """State for refund policy agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Refund Policy Agent

In [13]:
# System prompt for refund policy agent
REFUND_POLICY_PROMPT = """You are a helpful customer service agent specializing in REFUNDS, RETURNS, and EXCHANGES.

Your responsibilities:
- Explain refund policies clearly
- Guide customers through return procedures
- Process exchange requests
- Handle warranty claims
- Clarify eligibility requirements

Company policies (for demonstration):
- 30-day return window for most items
- Items must be unused and in original packaging
- Full refund issued within 5-7 business days
- Free return shipping for defective items
- Exchanges available for different sizes/colors
- Extended warranty available for electronics

Be empathetic, clear, and help resolve customer concerns professionally.
Always prioritize customer satisfaction while explaining policy constraints.
"""

def create_refund_policy_agent():
    """Create the refund policy agent"""

    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)

    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", REFUND_POLICY_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])

    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}

    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)

    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)

    return app

# Create the agent
refund_policy_agent = create_refund_policy_agent()
print("✅ Refund Policy Agent created successfully!")

✅ Refund Policy Agent created successfully!


## Test the Agent

In [14]:
# Test conversation
from uuid import uuid4
from openai import RateLimitError

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Test query 1
print("User: What's your refund policy?\n")

try:
    result = refund_policy_agent.invoke(
        {"messages": [HumanMessage(content="What's your refund policy?")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: Our refund policy allows returns within 30 days with proof of purchase.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)

User: What's your refund policy?

⚠️ API quota exceeded.
Agent: Our refund policy allows returns within 30 days with proof of purchase.

--------------------------------------------------------------------------------


In [15]:
# Test query 2 (with context)
from openai import RateLimitError

print("User: I bought this item 2 months ago, can I still return it?\n")

try:
    result = refund_policy_agent.invoke(
        {"messages": [HumanMessage(content="I bought this item 2 months ago, can I still return it?")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: Most standard return policies only allow returns within 30 days, so an item bought 2 months ago usually would not qualify unless there is an exception or warranty coverage.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)

User: I bought this item 2 months ago, can I still return it?

⚠️ API quota exceeded.
Agent: Most standard return policies only allow returns within 30 days, so an item bought 2 months ago usually would not qualify unless there is an exception or warranty coverage.

--------------------------------------------------------------------------------


In [16]:
# Test query 3 (follow-up)
from openai import RateLimitError

print("User: What if the item is defective?\n")

try:
    result = refund_policy_agent.invoke(
        {"messages": [HumanMessage(content="What if the item is defective?")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: If the item is defective, you may qualify for a refund, replacement, or warranty claim depending on the policy.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")

User: What if the item is defective?

⚠️ API quota exceeded.
Agent: If the item is defective, you may qualify for a refund, replacement, or warranty claim depending on the policy.

--------------------------------------------------------------------------------

✅ Agent maintained context across 3 turns!
